In [1]:
from google.colab import files
uploaded = files.upload()

Saving filtered3pt2003-2004 - Sheet1.csv to filtered3pt2003-2004 - Sheet1.csv
Saving filtered3pt2004-2005 - Sheet1.csv to filtered3pt2004-2005 - Sheet1.csv


In [3]:
import os
print("Uploaded Files:", [f for f in os.listdir('.') if f.endswith('.csv')])

Uploaded Files: ['filtered3pt2004-2005 - Sheet1.csv', 'filtered3pt2003-2004 - Sheet1.csv']


In [8]:
import pandas as pd

# 1. Load the exact file names uploaded in your current Colab environment
file_04 = "filtered3pt2003-2004 - Sheet1.csv"
file_05 = "filtered3pt2004-2005 - Sheet1.csv"

df_04 = pd.read_csv(file_04)
df_05 = pd.read_csv(file_05)

# 2. Strip hidden spaces from headers to prevent matching glitches
df_04.columns = [str(c).strip() for c in df_04.columns]
df_05.columns = [str(c).strip() for c in df_05.columns]

# 3. Robust Name Column Locator (finds a single column string)
name_cols_04 = [c for c in df_04.columns if 'Player' in c or 'Name' in c or c == 'clean_name']
name_cols_05 = [c for c in df_05.columns if 'Player' in c or 'Name' in c or c == 'clean_name']

# Grabs the first matched column string, or defaults to the first column name in the sheet
name_col_04 = name_cols_04[0] if name_cols_04 else df_04.columns[0]
name_col_05 = name_cols_05[0] if name_cols_05 else df_05.columns[0]

print(f"Using column '{name_col_04}' for 2003-04 file.")
print(f"Using column '{name_col_05}' for 2004-05 file.")

# 4. Standardize text inside the single player column
df_04['clean_name'] = df_04[name_col_04].astype(str).str.strip()
df_05['clean_name'] = df_05[name_col_05].astype(str).str.strip()

# 5. Clean up duplicate row entries or empty lines that inflate counts
df_04 = df_04[df_04['clean_name'].notna() & (df_04['clean_name'] != 'nan') & (df_04['clean_name'] != '')]
df_05 = df_05[df_05['clean_name'].notna() & (df_05['clean_name'] != 'nan') & (df_05['clean_name'] != '')]

df_04_unique = df_04.drop_duplicates(subset=['clean_name'])
df_05_unique = df_05.drop_duplicates(subset=['clean_name'])

# 6. Run the Inner Merge to match players side-by-side
matched_players = pd.merge(df_04_unique, df_05_unique, on="clean_name", suffixes=('_0304', '_0405'))

print(f"\n✅ Successfully matched {len(matched_players)} players who are in BOTH files!")
print("\nAvailable stats columns you can now compare:")
print([c for c in matched_players.columns if not c.endswith('_0405')][:15])


Using column 'Player' for 2003-04 file.
Using column 'Player' for 2004-05 file.

✅ Successfully matched 100 players who are in BOTH files!

Available stats columns you can now compare:
['Unnamed: 0_0304', 'Player_0304', 'Team_0304', 'Age_0304', 'GP_0304', 'W_0304', 'L_0304', 'Min_0304', 'PTS_0304', 'FGM_0304', 'FGA_0304', 'FG%_0304', '3:00 PM_0304', '3PA_0304', '3P%_0304']


In [10]:
# Select the exact side-by-side columns now that we know '3PM' is '3:00 PM'
cols = [
    'clean_name',
    'PTS_0304', 'PTS_0405',
    '3:00 PM_0304', '3:00 PM_0405',
    '3PA_0304', '3PA_0405',
    '3P%_0304', '3P%_0405'
]

# Create and view the clean comparison table
comparison_df = matched_players[cols].copy()
comparison_df.rename(columns={'clean_name': 'Player'}, inplace=True)
comparison_df



,Player,PTS_0304,PTS_0405,3:00 PM_0304,3:00 PM_0405,3PA_0304,3PA_0405,3P%_0304,3P%_0405
0,Jason Terry,16.8,12.4,1.8,1.3,5.2,3.1,34.7,42.0
1,Bob Sura,7.5,10.3,0.3,0.9,1.0,2.5,26.6,35.5
2,Wesley Person,5.8,6.4,0.9,1.0,2.4,2.2,39.9,46.1
3,Boris Diaw,4.5,4.8,0.1,0.1,0.3,0.8,23.1,18.0
4,Jacque Vaughn,3.8,5.3,0.0,0.1,0.3,0.2,15.0,33.3
...,...,...,...,...,...,...,...,...,...
95,Gordan Giricek,11.3,8.8,0.8,0.5,2.1,1.3,39.5,36.2
96,Raul Lopez,7.0,5.2,0.3,0.6,1.0,1.5,29.4,44.4
97,Gilbert Arenas,19.6,25.5,2.3,2.6,6.1,7.0,37.5,36.5
98,Larry Hughes,18.8,22.0,1.3,1.0,3.8,3.4,34.1,28.2
